# 记录要既够用，又不重复

沿用已有列关系，先判断删一列会不会丢失目标，再比较同一向量在两组完整基下的系数。两份坐标不同并不意味着向量移动了；基与顺序是坐标记录的一部分。完整换基与删掉一个系数也要分开。

从新内核按顺序运行；没有滑块时可直接修改参数。计算单元用于核对前面的推理，参考结果放在相应尝试之后。


In [ ]:
# 让图中的中文正常显示：先找本机已有的中文字体；找不到时（例如 Colab）下载思源黑体。
import pathlib
import urllib.request
import matplotlib.pyplot as plt
from matplotlib import font_manager

def use_chinese_font():
    candidates = ['Noto Sans CJK SC', 'Noto Sans SC', 'Source Han Sans SC', 'PingFang SC',
                  'Hiragino Sans GB', 'Microsoft YaHei', 'SimHei', 'WenQuanYi Zen Hei']
    installed = {font.name for font in font_manager.fontManager.ttflist}
    name = next((n for n in candidates if n in installed), None)
    if name is None:
        path = pathlib.Path.home() / '.cache' / 'linear-algebra-fonts' / 'NotoSansSC-Regular.otf'
        try:
            if not path.exists():
                path.parent.mkdir(parents=True, exist_ok=True)
                urllib.request.urlretrieve(
                    'https://github.com/notofonts/noto-cjk/raw/main/Sans/SubsetOTF/SC/'
                    'NotoSansSC-Regular.otf', path)
            font_manager.fontManager.addfont(str(path))
            name = font_manager.FontProperties(fname=str(path)).get_name()
        except Exception as error:
            print('没有找到中文字体，图中汉字可能显示为方框：', error)
            return
    plt.rcParams['font.family'] = [name, 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False

use_chinese_font()

In [ ]:
import numpy as np

e1 = np.array([1, 0])
e2 = np.array([0, 1])
d = np.array([1, 1])
z = np.array([2, 3])

with_d = 1 * e1 + 2 * e2 + 1 * d
without_d = 2 * e1 + 3 * e2
print('e1 + 2e2 + d =', with_d)
print('2e1 + 3e2   =', without_d)
print('两种写法都到达 z 吗？', np.array_equal(with_d, z) and np.array_equal(without_d, z))

## 系数可以换，结果保持不变

对目标 `(2, 3)`，一整族系数 `(2-t, 3-t, t)` 都能给出同一个向量，因为 `d=e1+e2`。拖动滑块，看系数变化；右侧的结果箭头应始终停在同一个位置。

In [ ]:
import matplotlib.pyplot as plt

def show_redundant_coordinates(t=0.0):
    coefficients = np.array([2 - t, 3 - t, t])
    result = coefficients[0] * e1 + coefficients[1] * e2 + coefficients[2] * d
    fig, (ax_coeff, ax_result) = plt.subplots(1, 2, figsize=(9, 4))
    ax_coeff.bar(['e1', 'e2', 'd'], coefficients,
                color=['#0072B2', '#D55E00', '#009E73'])
    ax_coeff.axhline(0, color='0.3', linewidth=0.8)
    ax_coeff.set(title='系数如何变化', ylabel='系数')
    ax_coeff.grid(axis='y', alpha=0.2)
    ax_result.quiver(0, 0, *result, angles='xy', scale_units='xy', scale=1,
                    color='#0072B2', width=0.012)
    ax_result.scatter(*z, marker='*', s=140, color='#D55E00', zorder=3)
    ax_result.set(xlim=(-0.5, 3), ylim=(-0.5, 4), xlabel='x', ylabel='y',
                  title='同一个目标向量')
    ax_result.set_aspect('equal', adjustable='box')
    ax_result.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
    print('系数 (e1, e2, d) =', coefficients)
    print('结果 =', result)

show_redundant_coordinates(t=0)  # 没有滑块时，修改 t 后重新运行

try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    print('未安装 ipywidgets，没有滑块；直接修改上面的函数参数也可以。')
else:
    display(widgets.interactive(
        show_redundant_coordinates,
        t=widgets.FloatSlider(value=0, min=-2, max=3, step=0.05, description='t')))

## 同一个向量，换一组基

目标改为 `z=(3,1)`。标准基 `e1,e2` 用系数 `(3,1)`；另一组 `p=(1,1)`、`q=(1,-1)` 会用什么系数？先算一算，再运行核对。

In [ ]:
p = np.array([1, 1])
q = np.array([1, -1])
target = np.array([3, 1])
standard_coordinates = np.array([3, 1])
alternate_coordinates = np.array([2, 1])
standard_result = standard_coordinates[0] * e1 + standard_coordinates[1] * e2
alternate_result = alternate_coordinates[0] * p + alternate_coordinates[1] * q
print('标准基下的坐标 =', standard_coordinates)
print('另一组基下的坐标 =', alternate_coordinates)
print('两组坐标都给出目标吗？', np.array_equal(standard_result, target) and
      np.array_equal(alternate_result, target))

如果一组向量既能拼出整个空间，又没有冗余，我们称它为一组基。不同基能给同一个向量不同的坐标；但在某一组基固定后，坐标不会再有第二套。试着用线性无关解释这个唯一性。